# Proyecto 3: Predicción de Riesgo de Diabetes
## Notebook 01 — Análisis Exploratorio de Datos (EDA)
**Autor:** Bryan Anthony López Guerrero | **Dataset:** Pima Indians Diabetes (UCI)

In [ ]:
!pip install pandas numpy matplotlib seaborn scipy -q
print('✓ Listo')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

C1='#E63946'; C2='#457B9D'; C3='#2D6A4F'; C4='#F4A261'
plt.rcParams.update({'axes.spines.top':False,'axes.spines.right':False,'figure.dpi':130})

# Cargar datos
from google.colab import files
uploaded = files.upload()  # sube diabetes.csv
import io
df = pd.read_csv(io.BytesIO(list(uploaded.values())[0]))
features = ['Pregnancies','Glucose','BloodPressure','SkinThickness',
            'Insulin','BMI','DiabetesPedigreeFunction','Age']
print(f'✓ Dataset: {df.shape}')
print(df.head())

In [ ]:
# Estadísticas descriptivas
print('ESTADÍSTICAS DESCRIPTIVAS')
print('='*60)
print(df.describe().round(2).to_string())
print(f"\nBalance de clases:")
print(df['Outcome'].value_counts().to_string())
print(f"\nPorcentaje positivos: {df['Outcome'].mean()*100:.1f}%")

In [ ]:
# Figura 1 — Balance de clases
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
counts = df['Outcome'].value_counts().sort_index()
labels = ['Sin diabetes\n(Outcome=0)', 'Con diabetes\n(Outcome=1)']
ax = axes[0]
bars = ax.bar(labels, counts.values, color=[C2,C1], width=0.5, alpha=0.9)
for bar, val in zip(bars, counts.values):
    pct = val/len(df)*100
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+5,
            f'{val}\n({pct:.1f}%)', ha='center', fontsize=11, fontweight='bold')
ax.set_title('Distribución de la variable objetivo', fontsize=12, fontweight='bold')
ax.set_ylabel('Número de pacientes'); ax.set_ylim(0,620); ax.grid(axis='y',alpha=0.3)

ax2 = axes[1]
ax2.pie(counts.values, labels=labels, colors=[C2,C1], autopct='%1.1f%%',
        startangle=90, wedgeprops={'edgecolor':'white','linewidth':2}, pctdistance=0.75)
ax2.set_title('Proporción de clases', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('01_balance_clases.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Figura 1 guardada')

In [ ]:
# Figura 2 — Distribuciones por clase
from scipy.stats import gaussian_kde
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()
for i, col in enumerate(features):
    ax = axes[i]
    for outcome, color, label in [(0,C2,'Sin diabetes'),(1,C1,'Con diabetes')]:
        data = df[df['Outcome']==outcome][col]
        data = data[data > 0]
        ax.hist(data, bins=25, alpha=0.6, color=color, label=label,
                density=True, edgecolor='white')
        if len(data) > 5:
            kde = gaussian_kde(data)
            x = np.linspace(data.min(), data.max(), 200)
            ax.plot(x, kde(x), color=color, lw=2)
    ax.set_title(col, fontsize=10, fontweight='bold')
    ax.set_ylabel('Densidad'); ax.grid(alpha=0.2)
    if i == 0: ax.legend(fontsize=8)
plt.suptitle('Distribución de variables clínicas por clase', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('02_distribuciones.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Figura 3 — Correlaciones
fig, ax = plt.subplots(figsize=(10, 8))
corr = df[features+['Outcome']].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, ax=ax, linewidths=0.5, square=True)
ax.set_title('Matriz de correlaciones', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('03_correlaciones.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Figura 4 — Boxplots
df_plot = df.copy()
for col in ['Glucose','BloodPressure','SkinThickness','Insulin','BMI']:
    df_plot.loc[df_plot[col]==0, col] = np.nan
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()
for i, col in enumerate(features):
    ax = axes[i]
    data0 = df_plot[df_plot['Outcome']==0][col].dropna()
    data1 = df_plot[df_plot['Outcome']==1][col].dropna()
    bp = ax.boxplot([data0,data1], patch_artist=True,
                    labels=['Sin\ndiabetes','Con\ndiabetes'],
                    medianprops={'color':'black','lw':2})
    bp['boxes'][0].set_facecolor(C2); bp['boxes'][0].set_alpha(0.7)
    bp['boxes'][1].set_facecolor(C1); bp['boxes'][1].set_alpha(0.7)
    ax.set_title(col, fontsize=10, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)
plt.suptitle('Boxplots por clase', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('04_boxplots.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Figura 5 — Valores cero imposibles
fig, ax = plt.subplots(figsize=(10,5))
cols_cero = ['Glucose','BloodPressure','SkinThickness','Insulin','BMI']
pct_ceros = [(df[c]==0).sum()/len(df)*100 for c in cols_cero]
n_ceros   = [(df[c]==0).sum() for c in cols_cero]
bars = ax.bar(cols_cero, pct_ceros,
              color=[C1 if p>10 else C4 if p>5 else C2 for p in pct_ceros],
              alpha=0.85, width=0.55)
for bar,n,p in zip(bars,n_ceros,pct_ceros):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
            f'{n} casos\n({p:.1f}%)', ha='center', fontsize=10)
ax.set_title('Valores cero clínicamente imposibles', fontsize=12, fontweight='bold')
ax.set_ylabel('% de registros con valor = 0')
ax.axhline(5, color='gray', linestyle='--', lw=1.2, label='Umbral 5%')
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('05_valores_cero.png', dpi=150, bbox_inches='tight')
plt.show()

from google.colab import files
for f in ['01_balance_clases.png','02_distribuciones.png','03_correlaciones.png',
          '04_boxplots.png','05_valores_cero.png']:
    files.download(f)
print('✓ Todas las figuras del EDA descargadas')